In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future

LAGS = [DateOffset(days=1),   DateOffset(days=2),   DateOffset(days=7),
        DateOffset(days=14),  DateOffset(days=21),  DateOffset(months=1),
        DateOffset(months=3), DateOffset(months=6), DateOffset(years=1)]

DIFFS = [DateOffset(days=1),   DateOffset(months=1),
         DateOffset(months=3), DateOffset(months=6)]

ROLL_WINDOWS = { 7: [DateOffset(days=1), DateOffset(days=7)],
                30: [DateOffset(months=1),
                     DateOffset(months=3),
                     DateOffset(months=6)]}


# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
df_train_store = attach_store_data(df_train, store_df)

# Shift Sales by 1 day per store to prevent any leakage of current-day sales into features
df_train_store = df_train_store.sort_values(['Store', 'Date'])
df_train_store['Shifted_Sales'] = df_train_store.groupby('Store')['Sales'].shift(1)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store.drop(['Sales'], axis=1),
                            lags=LAGS,
                            roll_windows=ROLL_WINDOWS,
                            diffs=DIFFS)
targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)


C:\Users\m_kal\AppData\Local\Temp\ipykernel_6292\855479055.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [3]:
pd.concat([
    df_features.dtypes,
    df_features.isna().sum()/len(df_features),
    df_features.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('feature_summary.csv')

In [4]:
df_features.shape, targets.shape

((1017209, 78), (1017209, 42))

# Hypertuning

In [ ]:
"""Nested time-series cross-validation with hyperopt for XGBForecaster."""

import numpy as np
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from typing import Callable

from src.xgb_forecaster import XGBForecaster
from src.engine import compute_metrics, cross_validate

import numpy as np
import xgboost as xgb
from src.engine import compute_metrics, cross_validate
from src.xgb_forecaster import XGBForecaster


def build_dataset():
    rng = np.random.default_rng(SEED)
    # Smooth random-walk features so XGBoost can learn something meaningful
    X_dummy = np.cumsum(rng.standard_normal((N_SAMPLES, N_FEATURES)), axis=0)

    # Targets: linear combination of features + tiny noise (multi-output)
    W = rng.standard_normal((N_FEATURES, HORIZON))
    y_dummy = X_dummy @ W + rng.standard_normal((N_SAMPLES, HORIZON)) * 0.1

    return X_dummy, y_dummy

SEARCH_SPACE: dict = {
    "n_estimators":    hp.choice("n_estimators",    [100, 200, 300, 500]),
    "max_depth":       hp.choice("max_depth",        [3, 4, 5, 6, 8]),
    "learning_rate":   hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
    "subsample":       hp.uniform("subsample",        0.6, 1.0),
    "colsample_bytree":hp.uniform("colsample_bytree", 0.5, 1.0),
    "min_child_weight":hp.choice("min_child_weight",  [1, 3, 5, 10, 20]),
    "reg_alpha":       hp.loguniform("reg_alpha",      np.log(1e-4), np.log(10.0)),
    "reg_lambda":      hp.loguniform("reg_lambda",     np.log(1e-4), np.log(10.0)),
}

# Default XGBoost parameters that are not being tuned
XGB_CONSTANTS: dict = {
    "early_stopping_rounds": 30,
    "tree_method": "hist",
    "device": "cuda", # "cpu"
    # multi_strategy="multi_output_tree" (vector leaf) is not yet supported on GPU;
    # omitting it uses the default "one_output_per_tree" which works on both CPU and GPU.
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "verbosity": 0,
}


SEED = 42
N_SAMPLES  = 5000   # number of time steps
N_FEATURES = 3      # number of input features
HORIZON    = 7      # forecast 7 steps ahead
NUM_OUTER_SPLITS = 3
NUM_INNER_SPLITS = 3
NUM_TRIALS = 2
BATCH_SIZE = 1024


def _make_objective(
    horizon: int,
    train_dm: xgb.DMatrix,
    n_inner_splits: int,
    batch_size: int | None,
    constant_params: dict,
) -> Callable:
    """Return a hyperopt objective function that runs inner-fold CV."""

    def objective(trial_hyperparams: dict) -> dict:

        # Combine trial hyperparameters with constants to form the full XGBoost params dict
        xgb_params = {**constant_params, **trial_hyperparams}
        forecaster = XGBForecaster(params=xgb_params,
                                   horizon=horizon,
                                   batch_size=batch_size)
        cv_result = cross_validate(forecaster, train_dm, n_splits=n_inner_splits)
        mean_mae = cv_result["mean"]["MAE"]

        # Return params in the "result" field of the best trial so they can be retrieved after fmin() 
        # finishes directly.
        return {"loss": mean_mae, "status": STATUS_OK, "params": xgb_params}

    return objective


# Build a single DMatrix for the full dataset — passed everywhere
X_dummy, y_dummy = build_dataset()
dm = xgb.DMatrix(X_dummy, label=y_dummy)

n = dm.num_row()
outer_tscv = TimeSeriesSplit(n_splits=NUM_OUTER_SPLITS)
results: list[dict] = []

for fold, (train_idx, test_idx) in enumerate(outer_tscv.split(np.arange(n)), start=1):
    print(f"\n{'='*60}")
    print(f"Outer fold {fold}/{NUM_OUTER_SPLITS}  "
        f"(train={len(train_idx)}, test={len(test_idx)})")
    print(f"{'='*60}")

    train_dm = dm.slice(train_idx)
    test_dm  = dm.slice(test_idx)

    # ── Inner loop: hyperopt ────────────────────────────────────────
    print(f"  Running hyperopt ({NUM_TRIALS} trials, {NUM_INNER_SPLITS} inner folds)...")
    objective = _make_objective(HORIZON, train_dm, NUM_INNER_SPLITS, BATCH_SIZE, XGB_CONSTANTS)
    trials = Trials()
    fmin(
        fn=objective,
        space=SEARCH_SPACE,
        algo=tpe.suggest,
        max_evals=NUM_TRIALS,
        trials=trials,
        verbose=False,
    )
    best_params = trials.best_trial["result"]["params"]

    print(f"  Best hyperparameters:")
    for k, v in best_params.items():
        if k not in ("device", "verbosity"):
            print(f"    {k}: {v:.6g}" if isinstance(v, float) else f"    {k}: {v}")

    # ── Retrain on full outer training fold ─────────────────────────
    print(f"  Retraining on full outer training fold...")
    final_forecaster = XGBForecaster(params=best_params,
                                     horizon=HORIZON,                      
                                     batch_size=BATCH_SIZE)
    final_forecaster.fit(train_dm)

    # ── Evaluate on outer test fold ─────────────────────────────────
    preds  = final_forecaster.predict(test_dm)
    y_test = test_dm.get_label()
    if HORIZON > 1:
        y_test = y_test.reshape(-1, HORIZON)
    metrics = compute_metrics(y_test, preds)

    print(f"  Outer fold {fold} test metrics:")
    for k, v in metrics.items():
        print(f"    {k}: {v:.4f}")

    results.append({
        "outer_fold":  fold,
        "best_params": best_params,
        "metrics":     metrics,
    })

# ── Summary ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("Nested CV summary (mean across outer folds):")
metric_keys = list(results[0]["metrics"].keys())
for k in metric_keys:
    vals = [r["metrics"][k] for r in results]
    print(f"  {k}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")


Outer fold 1/3  (train=1250, test=1250)
  Running hyperopt (2 trials, 3 inner folds)...
  Fold 1: MAE=10.8771, RMSE=14.6927, MAPE=71.8794, RMSPE=508.5587, R2=0.8566
  Fold 2: MAE=18.9934, RMSE=23.7380, MAPE=67.2320, RMSPE=508.6479, R2=0.9002
  Fold 3: MAE=4.4817, RMSE=5.9058, MAPE=78.1322, RMSPE=1180.1802, R2=0.9898
  Fold 1: MAE=10.9050, RMSE=14.9523, MAPE=75.8878, RMSPE=504.5259, R2=0.8515
  Fold 2: MAE=19.2937, RMSE=24.3351, MAPE=78.0467, RMSPE=637.0088, R2=0.8951
  Fold 3: MAE=13.7665, RMSE=16.4932, MAPE=161.9976, RMSPE=2001.2185, R2=0.9205
  Best hyperparameters:
    early_stopping_rounds: 30
    tree_method: hist
    objective: reg:squarederror
    eval_metric: rmse
    colsample_bytree: 0.968449
    learning_rate: 0.0169497
    max_depth: 5
    min_child_weight: 20
    n_estimators: 500
    reg_alpha: 0.000148417
    reg_lambda: 0.0421963
    subsample: 0.854039
  Retraining on full outer training fold...
  Outer fold 1 test metrics:
    MAE: 21.9892
    RMSE: 27.6046
    MAPE:

# TODOs:

- Check how the horizon is handled in the code internally.
- Check docs on batch size.
- Check the dates/stores coming out of the cv object.

In [3]:
train_size = int(0.8 * N_SAMPLES)
train_dm = dm.slice(list(range(train_size)))
test_dm  = dm.slice(list(range(train_size, N_SAMPLES)))

forecaster = XGBForecaster(
    horizon=HORIZON,
    params={"device": "cuda", "n_estimators": 500, "early_stopping_rounds": 30},
)
model = forecaster.fit(train_dm)
preds = forecaster.predict(test_dm)

y_test = test_dm.get_label().reshape(-1, HORIZON)
print("Hold-out metrics:")
hold_out_metrics = compute_metrics(y_test, preds)
for k, v in hold_out_metrics.items():
    print(f"  {k}: {v:.4f}")

Hold-out metrics:
  MAE: 24.2868
  RMSE: 29.9019
  MAPE: 29.8373
  RMSPE: 74.1343
  R2: 0.9664
